# Phase 2 — Data Pipeline Exploration

Run this notebook in **Google Colab** (T4 GPU) or locally after activating the ml-training venv.

Steps covered:
1. Download dataset from Kaggle
2. Run `data_preprocessing.py` to build train/val/test splits
3. Visualise class distribution
4. Preview augmented samples

## 0. Environment setup (Colab only)

In [ ]:
# Run this cell only when running on Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Mount Drive if you want to persist the dataset across sessions
    from google.colab import drive
    drive.mount('/content/drive')

    # Clone repo (or upload ml-training/ folder manually)
    # !git clone https://github.com/YOUR_REPO waste-classification-app
    # %cd waste-classification-app

    !pip install -q kaggle

## 1. Download Kaggle dataset

Place your `kaggle.json` API token at `~/.kaggle/kaggle.json` (or upload it in Colab).

In [ ]:
from pathlib import Path

RAW_DIR = Path('ml-training/data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Download only if raw dir is empty
if not any(RAW_DIR.iterdir()):
    !kaggle datasets download \
        -d alistairking/recyclable-and-household-waste-classification \
        -p {RAW_DIR} --unzip
    print('Download complete.')
else:
    print('Raw data already present — skipping download.')

## 2. Run preprocessing — build train/val/test splits

In [ ]:
!python ml-training/src/data_preprocessing.py

## 3. Class distribution

In [ ]:
import matplotlib.pyplot as plt

SPLITS_DIR = Path('ml-training/data/splits')
splits = ['train', 'val', 'test']

counts = {split: {} for split in splits}
for split in splits:
    split_path = SPLITS_DIR / split
    if not split_path.exists():
        continue
    for cls_dir in sorted(split_path.iterdir()):
        if cls_dir.is_dir():
            counts[split][cls_dir.name] = sum(1 for _ in cls_dir.rglob('*') if _.is_file())

classes = sorted(counts['train'].keys())
x = range(len(classes))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
for i, split in enumerate(splits):
    vals = [counts[split].get(c, 0) for c in classes]
    ax.bar([xi + i * width for xi in x], vals, width, label=split)

ax.set_xticks([xi + width for xi in x])
ax.set_xticklabels(classes, rotation=20, ha='right')
ax.set_ylabel('Image count')
ax.set_title('Class distribution across splits')
ax.legend()
plt.tight_layout()
plt.savefig('ml-training/outputs/class_distribution.png', dpi=150)
plt.show()
print('Saved to ml-training/outputs/class_distribution.png')

## 4. Preview augmented samples

In [ ]:
import sys
sys.path.insert(0, 'ml-training/src')

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from augmentation import build_augmentation_layer, build_preprocessing_layer

# Pick one sample image from train set
sample_cls = sorted(counts['train'].keys())[0]
sample_dir = SPLITS_DIR / 'train' / sample_cls
sample_path = next(sample_dir.glob('*'))

img = tf.keras.utils.load_img(sample_path, target_size=(256, 256))
img_arr = tf.keras.utils.img_to_array(img)
img_tensor = tf.expand_dims(img_arr, axis=0)  # (1, 256, 256, 3)

augment = build_augmentation_layer()

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes[0][0].imshow(img_arr.astype('uint8'))
axes[0][0].set_title('Original')
axes[0][0].axis('off')

for i, ax in enumerate(axes.flatten()[1:]):
    aug = augment(img_tensor, training=True)[0].numpy().clip(0, 255).astype('uint8')
    ax.imshow(aug)
    ax.set_title(f'Aug {i+1}')
    ax.axis('off')

fig.suptitle(f'Augmentation preview — {sample_cls}', fontsize=13)
plt.tight_layout()
plt.savefig('ml-training/outputs/augmentation_preview.png', dpi=150)
plt.show()
print('Saved to ml-training/outputs/augmentation_preview.png')